In [1]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import matplotlib as plt
import numpy as np

In [2]:
df = pd.read_csv("../data/DBScan/dbscan_clustered_0.17.csv")
df.shape

(1229374, 442)

In [3]:
cluster_modes = df[df['cluster_labels'] != -1].groupby('cluster_labels').agg(pd.Series.mode) # gets the mode of every value, similar to a centriod of kmodes 
cluster_means = df[df['cluster_labels'] != -1].groupby('cluster_labels').mean() #gets the prevalence of a feature per cluster
cluster_amount = len(set(df[df['cluster_labels'] != -1]["cluster_labels"]))
print("shape: ", df[df["cluster_labels"] != -1].shape)
print("number of clusters: ", cluster_amount)

shape:  (1220444, 442)
number of clusters:  8


In [4]:
cluster_modes.iloc[:,:5]

,properties_images,properties_notes,properties_orientation_data,properties_name,geometry_type
cluster_labels,,,,,
0,0.0,0.0,0.0,1.0,1.0
1,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,1.0,1.0
3,0.0,1.0,1.0,1.0,1.0
4,0.0,0.0,0.0,1.0,1.0
5,0.0,0.0,0.0,1.0,1.0
6,0.0,1.0,0.0,1.0,1.0
7,0.0,0.0,0.0,1.0,1.0


In [5]:
for cluster in cluster_modes.index:
    print(f"Cluster {cluster}: {df[df["cluster_labels"] == cluster].shape}")


Cluster 0: (1183288, 442)
Cluster 1: (12537, 442)
Cluster 2: (1359, 442)
Cluster 3: (6612, 442)
Cluster 4: (2991, 442)
Cluster 5: (7962, 442)
Cluster 6: (2307, 442)
Cluster 7: (3388, 442)


In [13]:
discriminating_features = []

for col in cluster_modes.columns:
    temp = set()
    for cluster in cluster_modes[col]:
        temp.add(cluster)
        
    if len(temp) > 1:
        discriminating_features.append(col)
       
print(f"{len(discriminating_features)} Discriminating Features: ") 
for feature in discriminating_features:
    print(f"\t{feature}")

51 Discriminating Features: 
	properties_notes
	properties_orientation_data
	geometry_type
	geometry_coordinates
	properties_trace_trace_feature
	properties_trace_trace_quality
	properties_trace_trace_type
	properties_trace_contact_type
	properties_trace_intrusive_contact_type
	properties_other_features
	properties_orientation_type
	properties_orientation_strike
	properties_orientation_dip_direction
	properties_orientation_dip
	properties_orientation_label
	properties_orientation_quality
	properties_orientation_feature_type
	properties_rock_unit_unit_label_abbreviation
	properties_rock_unit_rock_type
	properties_custom_fields_Station
	properties_custom_fields_STATNUM
	properties_custom_fields_VARIANT
	properties_custom_fields_GENERATION
	properties_custom_fields_UTMX
	properties_custom_fields_UTMY
	properties_custom_fields_OBJECTID
	properties_custom_fields_AREA
	properties_custom_fields_PERIMETER
	properties_custom_fields_GEO_
	properties_custom_fields_GEO_ID
	properties_custom_fields

In [7]:
print(cluster_means[discriminating_features])

                properties_notes  properties_orientation_data  geometry_type  \
cluster_labels                                                                 
0                       0.201990                     0.315801            1.0   
1                       0.044109                     0.411103            0.0   
2                       0.109639                     0.058131            1.0   
3                       1.000000                     1.000000            1.0   
4                       0.000000                     0.000000            1.0   
5                       0.392615                     0.000000            1.0   
6                       1.000000                     0.000000            1.0   
7                       0.039256                     0.000000            1.0   

                geometry_coordinates  properties_trace_trace_feature  \
cluster_labels                                                         
0                                1.0                   

In [8]:
global_mean = df[df['cluster_labels'] != -1].drop(columns='cluster_labels').mean() # presence in the entire dataset
deviation = cluster_means[discriminating_features] / global_mean[discriminating_features]

deviation

,properties_notes,properties_orientation_data,geometry_type,geometry_coordinates,properties_trace_trace_feature,properties_trace_trace_quality,properties_trace_trace_type,properties_trace_contact_type,properties_trace_intrusive_contact_type,properties_other_features,...,properties_custom_fields_GEO_MISC,properties_custom_fields_quality,properties_custom_fields_type,properties_rock_unit_era,properties_rock_unit_period,properties_rock_unit_epoch,properties_rock_unit_group_unit_type,properties_custom_fields_gid,properties_custom_fields_state,properties_custom_fields_county
cluster_labels,,,,,,,,,,,,,,,,,,,,,
0,0.978662,0.999712,1.010379,1.010379,0.954878,0.931644,0.954062,0.802742,0.173788,0.915795,...,0.000000,0.948995,0.691480,0.029936,0.032240,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.213715,1.301406,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.735154,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.531215,0.184022,1.010379,1.010379,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,4.845108,3.165643,1.010379,1.010379,0.000000,0.000000,0.000000,0.000000,0.000000,13.606666,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,1.010379,1.010379,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,408.038783,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,1.902262,0.000000,1.010379,1.010379,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,148.834634,148.492221,153.283597,153.283597,0.000000,0.000000,0.000000
6,4.845108,0.000000,1.010379,1.010379,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,529.017772,529.017772,529.017772
7,0.190201,0.000000,1.010379,1.010379,26.726026,34.840960,27.011133,79.860975,299.528494,0.000000,...,0.000000,28.780751,118.720233,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [9]:
significant_deviations = {}
cluster_defining_features = {}

for cluster in deviation.index:
    features = deviation.loc[cluster]
    significant = features[features >= 2].sort_values(ascending=False)
    significant_deviations[cluster] = significant

for cluster, features in significant_deviations.items():
    print(f"\nCluster {cluster} : {df[df["cluster_labels"] == cluster].shape[0]} samples")
    if len(features) == 0:
        print("\tNo significant features")
        cluster_defining_features[cluster] = []
    else:
        temp = []
        for feature, ratio in features.items():
            # print(f"\t{feature}: {ratio:.2f}x more present in cluster")
            print(f"\t{feature}")
            temp.append(feature)
        cluster_defining_features[cluster] = temp


Cluster 0 : 1183288 samples
	No significant features

Cluster 1 : 12537 samples
	properties_other_features

Cluster 2 : 1359 samples
	properties_orientation_quality
	properties_orientation_type
	properties_orientation_strike
	properties_orientation_dip_direction
	properties_orientation_dip
	properties_orientation_label
	properties_orientation_feature_type

Cluster 3 : 6612 samples
	properties_custom_fields_UTMX
	properties_custom_fields_UTMY
	properties_custom_fields_GENERATION
	properties_custom_fields_VARIANT
	properties_custom_fields_Station
	properties_custom_fields_STATNUM
	properties_other_features
	properties_notes
	properties_orientation_data

Cluster 4 : 2991 samples
	properties_custom_fields_GEO_MISC
	properties_custom_fields_PlateID
	properties_custom_fields_PERIMETER
	properties_custom_fields_GEO_
	properties_custom_fields_GEO_ID
	properties_custom_fields_GEO_TYPE
	properties_custom_fields_GEO_SYM
	properties_custom_fields_GEO_LINK
	properties_custom_fields_GEO_PAT
	proper

In [10]:
deviation[deviation.index == 0]

,properties_notes,properties_orientation_data,geometry_type,geometry_coordinates,properties_trace_trace_feature,properties_trace_trace_quality,properties_trace_trace_type,properties_trace_contact_type,properties_trace_intrusive_contact_type,properties_other_features,...,properties_custom_fields_GEO_MISC,properties_custom_fields_quality,properties_custom_fields_type,properties_rock_unit_era,properties_rock_unit_period,properties_rock_unit_epoch,properties_rock_unit_group_unit_type,properties_custom_fields_gid,properties_custom_fields_state,properties_custom_fields_county
cluster_labels,,,,,,,,,,,,,,,,,,,,,
0,0.978662,0.999712,1.010379,1.010379,0.954878,0.931644,0.954062,0.802742,0.173788,0.915795,...,0.0,0.948995,0.69148,0.029936,0.03224,0.0,0.0,0.0,0.0,0.0


In [16]:
def val_count(data):
    counter = 0
    for i in data.columns:
        
        # if i in ["properties_viewed_timestamp" ,"properties_modified_timestamp","properties_symbology_circleColor","properties_id","properties_symbology_lineColor","properties_symbology_lineWidth","properties_symbology_lineDasharray","properties_sed_strat_section_strat_section_id","properties_strat_section_id","properties_symbology_fillColor","properties_orientation_id","properties_custom_fields_osm_id","properties_orientation_modified_timestamp","properties_custom_fields_id","properties_orientation_unix_timestamp","properties_gps_accuracy","properties_altitude","properties_notesTimestamp"]: 
        
        print(counter)
        print(data[i].value_counts())
        counter += 1
        print("=========================="*5)
    print(data.shape)
    
val_count( df[df["cluster_labels"] == 0])

0
properties_images
0.0    1039086
1.0     144202
Name: count, dtype: int64
1
properties_notes
0.0    944276
1.0    239012
Name: count, dtype: int64
2
properties_orientation_data
0.0    809605
1.0    373683
Name: count, dtype: int64
3
properties_name
1.0    1181168
0.0       2120
Name: count, dtype: int64
4
geometry_type
1.0    1183288
Name: count, dtype: int64
5
geometry_coordinates
1.0    1183288
Name: count, dtype: int64
6
properties_samples
0.0    1153058
1.0      30230
Name: count, dtype: int64
7
properties_altitude_accuracy
0.0    1162839
1.0      20449
Name: count, dtype: int64
8
properties_lng
0.0    1159917
1.0      23371
Name: count, dtype: int64
9
properties_image_basemap
0.0    1156726
1.0      26562
Name: count, dtype: int64
10
properties_lat
0.0    1159918
1.0      23370
Name: count, dtype: int64
11
properties__3d_structures
0.0    1179147
1.0       4141
Name: count, dtype: int64
12
properties_trace_trace_feature
0.0    1141011
1.0      42277
Name: count, dtype: int64
13
